# ZelusBench: Attention Complex

**Stress test: all difficulty knobs at maximum.**

Deep chains (16-32), diamond DAG, extreme distractors,
heavy transforms, 3D, all query/point types.
Designed to be very challenging.

- 15 scenarios, 75 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response




SEEDS = 15
print(f"ZelusBench Attention Complex")
print(f"Seeds: {SEEDS} | Total: {SEEDS} scenarios")

In [ ]:
@kbench.task(name="zelusbench_attn_complex")
def zelusbench_attn_complex(llm) -> tuple[float, float]:
    """Combined complex: all knobs at maximum difficulty.
    Returns (mean_accuracy, std_dev)."""

    _start = time.time()
    print(f"Running {SEEDS} complex scenarios...")
    print("=" * 60)

    all_scores = []

    for seed in range(SEEDS):
        rng = random.Random(seed)
        density = rng.choice([TransformDensity.HEAVY, TransformDensity.EXTREME])
        cfg = ScenarioConfig(
            dim=3,
            min_chain_depth=16, max_chain_depth=32,
            dag_structure=DAGStructure.DIAMOND,
            distractor_level=DistractorLevel.EXTREME,
            transform_density=density,
            transform_types=["rotation", "translation", "reflection", "scaling"],
            query_types=[QueryType.POSITION, QueryType.DISTANCE, QueryType.BOOLEAN],
            point_def_types=[
                "cartesian_offset", "magnitude_direction", "magnitude_polar",
                "magnitude_spherical", "midpoint", "weighted_centroid",
            ],
            coord_min=-20.0, coord_max=20.0,
            magnitude_min=2.0, magnitude_max=15.0,
            num_queries=5, num_splits=5,
            seed=seed,
        )
        gen = ScenarioGenerator(cfg)
        scenario = gen.generate(f"complex_s{seed}")

        response = llm.prompt(scenario.prompt)
        scores = score_response(response, scenario)
        all_scores.extend(scores)

        avg = float(np.mean([s.score for s in scores]))
        q_depths = [s.chain_depth for s in scores]
        tiers = [s.tier.name for s in scores]
        print(f"  [{seed+1}/{SEEDS}] tx={density.name} avg={avg:.2f} q_depths={q_depths} tiers={tiers}")

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    kbench.assertions.assert_true(
        overall >= 0,
        expectation=f"Complex: model should produce valid answers (got {overall:.3f})"
    )

    return overall, std


zelusbench_attn_complex.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_complex